# UNSW-NB15 — Exploratory Data Analysis

**Goal of this notebook:**
- Understand what the UNSW-NB15 dataset contains
- Identify class imbalance (attack vs normal, per attack category)
- Profile key feature distributions and spot anomalies
- Validate that the preprocessing pipeline produces sensible output
- Inform model design decisions (e.g. which features matter, how severe the imbalance is)

**Run AFTER:**
```
python data/download_data.py
python run_preprocessing.py
```

In [ ]:
import sys, json
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from src.preprocessing import load_splits, DataCleaner, ATTACK_CATEGORIES

# Notebook-wide aesthetics
sns.set_theme(style='darkgrid', palette='muted', font_scale=1.2)
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['figure.dpi'] = 120

PROCESSED = Path.cwd().parent / 'data' / 'processed'
print('Processed data directory:', PROCESSED)

## 1. Load Preprocessing Report

In [ ]:
report_path = PROCESSED / 'preprocessing_report.json'
with open(report_path) as f:
    report = json.load(f)

print(f"Train rows       : {report['train_rows']:,}")
print(f"Test rows        : {report['test_rows']:,}")
print(f"Normal-only rows : {report['normal_only_rows']:,}")
print(f"Feature count    : {report['n_features']}")
print(f"Normal:Attack ratio (train): {report['imbalance_ratio']:.2f}:1")
print(f"\nAttack categories: {report['attack_cat_classes']}")

In [ ]:
# Load the processed splits
train_df, test_df, normal_df = load_splits(PROCESSED)
print(f'Train: {train_df.shape} | Test: {test_df.shape} | Normal-only: {normal_df.shape}')
train_df.head(3)

## 2. Class Imbalance Analysis

**Why this matters:** Class imbalance is the most critical data issue for intrusion detection systems.
If normal traffic is 80% and one attack type is 0.1%, a model that predicts 'normal' for everything
gets 80%+ accuracy while being useless. We need precision/recall/F1 per class, not accuracy.

We'll also check if our supervised models need class-weight balancing or SMOTE.

In [ ]:
# Binary label distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Binary
binary_counts = train_df['label'].value_counts().rename({0: 'Normal', 1: 'Attack'})
axes[0].bar(binary_counts.index, binary_counts.values,
            color=['#2ecc71', '#e74c3c'], edgecolor='black', linewidth=0.8)
axes[0].set_title('Binary Label Distribution (Train)', fontweight='bold')
axes[0].set_ylabel('Sample Count')
for i, v in enumerate(binary_counts.values):
    axes[0].text(i, v + 200, f'{v:,}\n({100*v/len(train_df):.1f}%)',
                ha='center', fontsize=10)

# Attack category
cat_counts = train_df['attack_cat'].value_counts()
colors = sns.color_palette('husl', len(cat_counts))
bars = axes[1].barh(cat_counts.index, cat_counts.values, color=colors, edgecolor='black', linewidth=0.5)
axes[1].set_title('Attack Category Distribution (Train)', fontweight='bold')
axes[1].set_xlabel('Sample Count')
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
for bar, count in zip(bars, cat_counts.values):
    axes[1].text(count + 100, bar.get_y() + bar.get_height()/2,
                f'{count:,} ({100*count/len(train_df):.2f}%)',
                va='center', fontsize=9)

plt.tight_layout()
plt.savefig(PROCESSED.parent.parent / 'reports' / 'eda_class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('\nKey insight: note the severe imbalance between categories (e.g., Exploits vs Worms)')

## 3. Feature Distribution Analysis

We'll look at a subset of the most informative features — those with high variance
or strong correlation with the attack label.

In [ ]:
# Feature columns (exclude targets)
cleaner = DataCleaner.load(PROCESSED / 'cleaner.joblib')
feat_cols = cleaner.feature_cols
print(f'Total features: {len(feat_cols)}')
print(feat_cols)

In [ ]:
# Compute point-biserial correlation of each feature with the binary label
# (equivalent to Pearson for binary target — good quick importance proxy)
from scipy.stats import pointbiserialr

correlations = {}
for col in feat_cols:
    if col in train_df.columns:
        r, p = pointbiserialr(train_df['label'], train_df[col])
        correlations[col] = abs(r)  # absolute correlation

corr_series = pd.Series(correlations).sort_values(ascending=False)

# Plot top 20 features by correlation with label
fig, ax = plt.subplots(figsize=(12, 8))
top20 = corr_series.head(20)
colors = plt.cm.RdYlGn_r(np.linspace(0.2, 0.8, len(top20)))
ax.barh(top20.index[::-1], top20.values[::-1], color=colors[::-1], edgecolor='black', linewidth=0.5)
ax.set_xlabel('|Point-Biserial Correlation| with Binary Label', fontsize=12)
ax.set_title('Top 20 Features Correlated with Attack Label', fontweight='bold', fontsize=14)
ax.axvline(0.1, color='red', linestyle='--', alpha=0.5, label='|r|=0.1 threshold')
ax.legend()
plt.tight_layout()
plt.savefig(PROCESSED.parent.parent / 'reports' / 'eda_feature_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Distribution of top 6 features — Normal vs Attack overlay
top6 = corr_series.head(6).index.tolist()
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
axes = axes.flatten()

normal_mask = train_df['label'] == 0
attack_mask = train_df['label'] == 1

for i, col in enumerate(top6):
    axes[i].hist(train_df.loc[normal_mask, col], bins=60, alpha=0.6,
                 color='#2ecc71', label='Normal', density=True)
    axes[i].hist(train_df.loc[attack_mask, col], bins=60, alpha=0.6,
                 color='#e74c3c', label='Attack', density=True)
    axes[i].set_title(col, fontweight='bold')
    axes[i].legend(fontsize=9)
    axes[i].set_ylabel('Density')

plt.suptitle('Feature Distributions: Normal vs Attack (Top 6)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(PROCESSED.parent.parent / 'reports' / 'eda_feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Feature Correlation Heatmap (Top Features)

Looking for multicollinearity — if two features are perfectly correlated, one is redundant.
Tree-based models (RF, XGBoost) handle this automatically, but the LSTM-Autoencoder's
reconstruction error can be inflated by redundant features. Good to know.

In [ ]:
top15 = corr_series.head(15).index.tolist()
corr_matrix = train_df[top15].corr()

fig, ax = plt.subplots(figsize=(13, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))  # only lower triangle
sns.heatmap(
    corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
    center=0, vmin=-1, vmax=1, ax=ax,
    cbar_kws={'shrink': 0.8}, annot_kws={'size': 8}
)
ax.set_title('Feature Correlation Heatmap (Top 15 by Label Correlation)', fontweight='bold')
plt.tight_layout()
plt.savefig(PROCESSED.parent.parent / 'reports' / 'eda_correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Attack Category Profiles

For each attack category, compute mean feature values (relative to normal baseline).
This informs the MITRE ATT&CK mapping rationale — e.g. DoS attacks have very high Spkts/Dpkts.

In [ ]:
# Mean feature values per category (normalized, so 0=global mean)
top8_features = corr_series.head(8).index.tolist()
category_profiles = train_df.groupby('attack_cat')[top8_features].mean()

fig, ax = plt.subplots(figsize=(16, 7))
sns.heatmap(
    category_profiles,
    annot=True, fmt='.2f', cmap='coolwarm',
    center=0, ax=ax,
    cbar_kws={'label': 'Mean (Standardized)'}
)
ax.set_title('Attack Category Feature Profiles (Mean Standardized Values)', fontweight='bold', fontsize=13)
ax.set_xlabel('Feature')
ax.set_ylabel('Attack Category')
plt.tight_layout()
plt.savefig(PROCESSED.parent.parent / 'reports' / 'eda_category_profiles.png', dpi=150, bbox_inches='tight')
plt.show()
print('\nNote: These profiles help justify the MITRE ATT&CK technique mappings in Phase 5.')

## 6. Normal-Only Distribution (for Unsupervised Model Training)

The unsupervised models (Isolation Forest, LSTM-Autoencoder) are trained ONLY on this subset.
It's important to verify: (a) it's all actually normal, (b) it's large enough for training.

In [ ]:
print(f'Normal-only subset: {len(normal_df):,} rows')
print(f'Label values in normal subset: {normal_df["label"].unique()}  ← should be only [0]')
print(f'Attack cats in normal subset: {normal_df["attack_cat"].unique()}  ← should be only Normal')

# Feature statistics for normal traffic
normal_stats = normal_df[top8_features].describe()
print(f'\nNormal traffic feature statistics (top 8 features):')
normal_stats.style.background_gradient(cmap='Blues')

## 7. EDA Summary & Key Findings

Fill this in after running the cells above.

In [ ]:
# Print a structured summary to include in your project report
print('='*60)
print('EDA SUMMARY — Key Findings')
print('='*60)
print(f'Dataset size: {len(train_df)+len(test_df):,} total samples')
print(f'Features: {len(feat_cols)} after dropping IPs, timestamps, and metadata')
print()
print('Class imbalance:')
for cat, count in sorted(train_df['attack_cat'].value_counts().items(), key=lambda x: -x[1]):
    pct = 100*count/len(train_df)
    bar = '█' * int(pct/2)
    print(f'  {cat:<18} {count:>8,}  ({pct:5.2f}%)  {bar}')
print()
print(f'Most predictive features (top 5):')
for feat, corr in corr_series.head(5).items():
    print(f'  {feat:<20} |r| = {corr:.3f}')
print()
print('Design implications:')
print('  → Use class_weight="balanced" for RF/XGBoost')
print('  → Evaluate with per-category F1, not overall accuracy')
print('  → Rare categories (Worms, Shellcode) will have high uncertainty')
print('  → Normal-only training set is large enough for unsupervised models')